In [ ]:
import pandas as pd
from transformers import AutoTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from transformers import TrainingArguments
from transformers import AutoModelForSequenceClassification
import torch

In [ ]:
from datasets import load_dataset
dataset = load_dataset('mteb/RuReviewsClassification')

In [ ]:
print(dataset)
print(dataset['train'].features)

for label in [0, 1, 2]:
    print(f"=== Label {label} ===")
    examples = [x for x in dataset['train'] if x['label'] == label][:8]
    for elem in examples:
        print(elem['text'][:200])
        print('---')
    print()

In [6]:
label_names = {0: 'negative', 1: 'neutral', 2: 'positive'}

In [8]:
labels = dataset['train']['label']
counts = pd.Series(labels).value_counts().sort_index()
counts.index = counts.index.map(label_names)
print(counts)

negative    15000
neutral     15000
positive    15000
Name: count, dtype: int64


## Токенизация данных

In [13]:
tokenizer = AutoTokenizer.from_pretrained('cointegrated/rubert-tiny2')

def tokenize_function(examples):
    return tokenizer(
        examples['text'],
        truncation=True,
        padding='max_length',
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map: 100%|██████████| 2048/2048 [00:00<00:00, 30563.40 examples/s]


In [14]:
print(tokenized_dataset)
print(tokenized_dataset['train'][0].keys())
print(len(tokenized_dataset['train'][0]['input_ids']))

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 45000
    })
    validation: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 15000
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 2048
    })
})
dict_keys(['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask'])
128


## Baseline: TF-IDF + LogisticRegression

In [16]:
train_texts = dataset['train']['text']
train_labels = dataset['train']['label']

test_texts = dataset['test']['text']
test_labels = dataset['test']['label']

vectorizer = TfidfVectorizer(max_features=10000)
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, train_labels)

baseline_preds = baseline_model.predict(X_test)
baseline_f1 = f1_score(test_labels, baseline_preds, average='macro')

print(f"Baseline F1 (macro): {baseline_f1:.4f}")
print(classification_report(test_labels, baseline_preds, target_names=['negative', 'neutral', 'positive']))

Baseline F1 (macro): 0.7353
              precision    recall  f1-score   support

    negative       0.72      0.70      0.71       682
     neutral       0.62      0.64      0.63       683
    positive       0.87      0.86      0.86       683

    accuracy                           0.73      2048
   macro avg       0.74      0.73      0.74      2048
weighted avg       0.74      0.73      0.74      2048



In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'cointegrated/rubert-tiny2',
    num_labels=3
)

Loading weights: 100%|██████████| 55/55 [00:00<00:00, 36698.49it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the c

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    logging_strategy='steps',
    logging_steps=100,
)


In [33]:
import numpy as np
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    f1 = f1_score(labels, predictions, average='macro')
    accuracy = accuracy_score(labels, predictions)
    return {'f1': f1, 'accuracy': accuracy}

In [34]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    compute_metrics=compute_metrics,
)

In [35]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1,Accuracy
1,0.584700,0.568696,0.749042,0.749067
2,0.545885,0.553009,0.761661,0.759333
3,0.494421,0.556426,0.763069,0.761600


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.74it/s]


TrainOutput(global_step=4221, training_loss=0.5592006560440623, metrics={'train_runtime': 70.1005, 'train_samples_per_second': 1925.808, 'train_steps_per_second': 60.214, 'total_flos': 248911937280000.0, 'train_loss': 0.5592006560440623, 'epoch': 3.0})